# SAE vs Raw, With a Dictionary That Is Actually Trained

## What went wrong in `60`, in one line

**5,120 latents, 122,031 training tokens — about 24 tokens per latent.** The dictionary collapsed
onto ~77 live latents (98.5% dead) and the comparison was, by its own pre-registered gate,
**inconclusive**.

| `60` | value |
|---|---|
| FVU (held out) | 0.094 / 0.117 — *reconstruction was fine* |
| mean L0 | 32.0 — *sparsity mechanism worked* |
| **dead latents** | **98.5% / 94.2%** — *the dictionary was not trained* |

## The three fixes

**1. Get the corpus from real proteins, not generated ones.**
This is the change that matters. `60` spent ~30 min generating 6,000 sequences to obtain 122k
tokens, because ProtGPT2 at `max_length=50` emits only ~23 tokens per sequence. But **an SAE does
not need generated text — it needs activations**, and a forward pass is far cheaper than
autoregressive generation. Fetching real UniProt proteins (300+ residues each) and running forward
passes gives **~6× the tokens in a fraction of the time.**

**2. A dictionary width the corpus can support.** 2,560 latents (2× expansion) instead of 5,120.

**3. AuxK dead-latent revival** (Gao et al.). A latent only learns when it wins a TopK slot; one
that never wins never receives a gradient and stays dead forever. AuxK takes the *dead* latents and
makes their top-`k_aux` reconstruct the **residual** the main `k` left behind, so they get a
gradient and come back.

| | `60` | this notebook |
|---|---|---|
| corpus source | 6,000 **generated** sequences | ~12,000 **real UniProt** proteins |
| training tokens | 122,031 | **~700k** target |
| latents | 5,120 | **2,560** |
| tokens per latent | **~24** | **~275** |
| dead-latent handling | none | **AuxK revival** |

Still below what a dedicated SAE paper would use — this is a workshop-scale dictionary and the
notebook says so — but roughly **10× better resourced**, and the health metrics decide whether it
is admissible.

## Everything else is held identical to `60`

Matched feature budget (k ∈ {3, 20, 64} on both sides), matched layer (12 and 30), matched
classifier, latent selection fitted **inside each CV fold**, and the same pre-registered health gate
(`FVU < 0.5` **and** `dead_frac < 0.9`). The labelled probe set is regenerated with `60`'s seed
(70001) so the two runs are directly comparable.

## What this can and cannot be

**It is a replication.** Kantamneni et al. (ICML 2025, arXiv:2502.16681) own the
SAE-underperforms-baselines result, established across >100 datasets and 8 SAE architectures.
The best outcome here is *"the same holds on a protein language model with a structural target."*
**Do not write it as a discovery.**

**Why not use a pre-trained SAE?** InterPLM (Simon & Zou, *Nature Methods* 2025) released SAEs for
protein LMs — but they are trained on **ESM-2**, and **ESMFold's language trunk is ESM-2**. Using
ESM-2 activations to predict ESMFold's pLDDT means reading the evaluator's own internal
representation to predict the evaluator's verdict. It would score well and mean nothing. No
pre-trained SAE exists for ProtGPT2, and dictionaries do not transfer across activation spaces.

## Pre-registered readings

| outcome | reading |
|---|---|
| health gate passes, **SAE < raw** | The replication lands. Report as a controlled replication of Kantamneni et al. in a new domain; cite them for the direction. |
| health gate passes, **SAE ≈ raw** | At matched budget the two are equivalent here. Also publishable, and a cleaner correction of §3a. |
| health gate passes, **SAE > raw** | Contrary to ICML. Check in-fold selection and corpus disjointness before believing it. |
| **health gate fails again** | Report inconclusive a second time and **stop** — two honest failures to build an adequate dictionary at workshop scale is itself the finding, and it is time to draft. |

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect **~1.5–2 hours**
(UniProt fetch, forward passes, two SAE trainings, 800 labelled generations + folds).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from scipy.stats import fisher_exact, mannwhitneyu

torch.manual_seed(2026)
np.random.seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ProtGPT2's own natural v_L norm from notebook 03. Used ONLY to define the anchor and to report
# how far the old absolute-norm runs were from matched -- never to scale anything in this notebook.
REFERENCE_NORM = 583.998

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())
print("Model under test: nferruz/ProtGPT2")


Setup complete. CUDA available: True
Model under test: nferruz/ProtGPT2


In [2]:
# --- Config. ---

import torch.nn as nn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score

LAYERS = [12, 30]
K_VALUES = [3, 20, 64]
N_UNIPROT = 12000          # real proteins for the SAE corpus
MAX_TOK_PER_SEQ = 64       # cap tokens kept per protein (memory)
D_LATENT = 2560            # 2x expansion on 1280 -- what ~700k tokens can support
SAE_TOPK = 32
AUX_K = 256                # dead latents given a shot at the residual
N_LABELLED = 800
N_REPEATS = 10
LABELLED_SEED = 70001      # same as 60, so the probe sets match

print(f"target corpus: {N_UNIPROT} proteins x <= {MAX_TOK_PER_SEQ} tokens")
print(f"dictionary: {D_LATENT} latents, TopK={SAE_TOPK}, AuxK={AUX_K}")
print(f"60 had 5120 latents on 122k tokens = 24 tokens/latent and went 98.5% dead")


target corpus: 12000 proteins x <= 64 tokens
dictionary: 2560 latents, TopK=32, AuxK=256
60 had 5120 latents on 122k tokens = 24 tokens/latent and went 98.5% dead


In [3]:
# --- Bulk UniProt fetch. This replaces 60's 30 minutes of generation, and yields ~6x the tokens.
#     Cursor pagination via the Link header is UniProt's documented method for large downloads. ---

import urllib.request
import urllib.error
import re as _re

def fetch_uniprot_bulk(n_target, page=500, min_len=120, max_len=400):
    base = ("https://rest.uniprot.org/uniprotkb/search"
            f"?query=reviewed:true+AND+length:%5B{min_len}+TO+{max_len}%5D"
            f"&format=fasta&size={page}")
    seqs, url, pages = [], base, 0
    while url and len(seqs) < n_target:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "ai4dd-sae/1.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                body = r.read().decode("utf-8")
                link = r.headers.get("Link", "")
        except Exception as e:
            print(f"  !! fetch failed on page {pages}: {type(e).__name__}: {e}")
            break
        cur, buf = None, []
        for line in body.split("\n"):
            if line.startswith(">"):
                if buf:
                    seqs.append("".join(buf))
                buf = []
            elif line.strip():
                buf.append(line.strip())
        if buf:
            seqs.append("".join(buf))
        pages += 1
        m = _re.search(r'<([^>]+)>;\s*rel="next"', link)
        url = m.group(1) if m else None
        if pages % 5 == 0:
            print(f"  page {pages}: {len(seqs)} sequences")
    return seqs[:n_target]

print("Fetching real UniProt proteins for the SAE corpus...")
corpus_seqs = fetch_uniprot_bulk(N_UNIPROT)
VALID_AA_S = set("ACDEFGHIKLMNPQRSTVWY")
corpus_seqs = ["".join(a for a in s if a in VALID_AA_S) for s in corpus_seqs]
corpus_seqs = [s for s in corpus_seqs if len(s) >= 100]

USED_UNIPROT = len(corpus_seqs) >= 2000
print(f"\n{len(corpus_seqs)} usable proteins, mean length "
      f"{np.mean([len(s) for s in corpus_seqs]):.0f} residues")
if not USED_UNIPROT:
    print("!" * 78)
    print("!! FETCHED TOO FEW PROTEINS -- Kaggle Internet is probably OFF, or the UniProt API")
    print("!! changed. Without a real corpus this notebook cannot train an adequate dictionary")
    print("!! and will repeat 60's failure. Fix Internet and re-run rather than continuing.")
    print("!" * 78)


Fetching real UniProt proteins for the SAE corpus...
  page 5: 2500 sequences
  page 10: 5000 sequences
  page 15: 7500 sequences
  page 20: 10000 sequences

12000 usable proteins, mean length 272 residues


In [4]:
# --- Token activations from the real-protein corpus. One layer at a time, to keep RAM sane. ---

print(f"Loading ProtGPT2 on {device}...")
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()
PAD_ID = tokenizer.eos_token_id
D_MODEL = plm_model.config.n_embd

rng = np.random.RandomState(11)

def token_acts_for_layer(seqs, layer, max_tok):
    chunks, ntok = [], 0
    for i, s in enumerate(seqs):
        enc = tokenizer(s, return_tensors="pt", truncation=True, max_length=320).to(device)
        with torch.no_grad():
            out = plm_model(**enc, output_hidden_states=True)
        h = out.hidden_states[layer].squeeze(0).float().cpu()
        if h.shape[0] > max_tok:
            h = h[rng.choice(h.shape[0], max_tok, replace=False)]
        chunks.append(h.half())
        ntok += h.shape[0]
        if (i + 1) % 2000 == 0:
            print(f"    {i + 1}/{len(seqs)} proteins, {ntok} tokens")
    return torch.cat(chunks, dim=0)

def clean_aa(text):
    return "".join(a for a in text.replace(" ", "") if a in VALID_AA)

print(f"d_model={D_MODEL}")


Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


d_model=1280


In [5]:
# --- The SAE: TopK with AuxK dead-latent revival. ---

class TopKSAE(nn.Module):
    def __init__(self, d_model, d_latent, k, aux_k):
        super().__init__()
        self.k, self.aux_k, self.d_latent = k, aux_k, d_latent
        self.enc = nn.Linear(d_model, d_latent)
        self.dec = nn.Linear(d_latent, d_model, bias=False)
        self.b_pre = nn.Parameter(torch.zeros(d_model))
        with torch.no_grad():
            self.dec.weight.copy_(self.enc.weight.t())
        # steps since each latent last fired; used to identify dead ones
        self.register_buffer("last_fired", torch.zeros(d_latent))

    def preacts(self, x):
        return F.relu(self.enc(x - self.b_pre))

    def encode(self, x):
        a = self.preacts(x)
        v, i = torch.topk(a, self.k, dim=-1)
        return torch.zeros_like(a).scatter_(-1, i, v)

    def forward(self, x, dead_mask=None):
        a = self.preacts(x)
        v, i = torch.topk(a, self.k, dim=-1)
        z = torch.zeros_like(a).scatter_(-1, i, v)
        recon = self.dec(z) + self.b_pre
        aux_recon = None
        if dead_mask is not None and dead_mask.any():
            # AuxK: let the DEAD latents try to explain what the main k left behind.
            # Without this a latent that never wins a TopK slot never gets a gradient and can
            # never recover -- which is exactly how 60 ended up 98.5% dead.
            a_dead = a.masked_fill(~dead_mask.unsqueeze(0), 0.0)
            kk = min(self.aux_k, int(dead_mask.sum().item()))
            if kk > 0:
                dv, di = torch.topk(a_dead, kk, dim=-1)
                z_aux = torch.zeros_like(a).scatter_(-1, di, dv)
                aux_recon = self.dec(z_aux)
        return recon, z, aux_recon, i

def train_sae(X, d_latent, k, aux_k, epochs=10, bs=4096, lr=3e-4, seed=0,
              val_frac=0.08, aux_coef=1.0 / 32, dead_after=2_000_000):
    torch.manual_seed(seed)
    sae = TopKSAE(X.shape[1], d_latent, k, aux_k).to(device)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    shuf = torch.randperm(X.shape[0])
    n_val = max(1, min(20000, int(val_frac * X.shape[0])))
    Xv = X[shuf[:n_val]].float().to(device)
    Xt = X[shuf[n_val:]]
    n = Xt.shape[0]
    seen = 0
    print(f"    train {n} tokens / val {n_val} held out  ({n / d_latent:.0f} tokens per latent)")
    for ep in range(epochs):
        perm = torch.randperm(n)
        tot, nb_ = 0.0, 0
        for s0 in range(0, n, bs):
            xb = Xt[perm[s0:s0 + bs]].float().to(device)
            dead = (seen - sae.last_fired) > dead_after
            recon, z, aux_recon, idx = sae(xb, dead_mask=dead if dead.any() else None)
            loss = F.mse_loss(recon, xb)
            if aux_recon is not None:
                loss = loss + aux_coef * F.mse_loss(aux_recon, (xb - recon).detach())
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                sae.dec.weight.div_(sae.dec.weight.norm(dim=0, keepdim=True) + 1e-8)
                seen += xb.shape[0]
                sae.last_fired[idx.reshape(-1).unique()] = float(seen)
            tot += loss.item(); nb_ += 1
        with torch.no_grad():
            r, z, _, _ = sae(Xv)
            fvu = ((Xv - r) ** 2).sum() / ((Xv - Xv.mean(0)) ** 2).sum()
            dead_now = float((z.abs().sum(0) == 0).float().mean())
        print(f"    epoch {ep + 1}/{epochs}  loss {tot / nb_:.4f}  FVU {fvu.item():.4f}  "
              f"dead {dead_now:.1%}")
    with torch.no_grad():
        r, z, _, _ = sae(Xv)
        fvu = float(((Xv - r) ** 2).sum() / ((Xv - Xv.mean(0)) ** 2).sum())
        l0 = float((z > 0).float().sum(-1).mean())
        dead = float((z.abs().sum(0) == 0).float().mean())
    assert math.isfinite(fvu), "FVU not finite -- training diverged"
    assert abs(l0 - k) < max(1.0, 0.1 * k), f"mean L0 {l0:.2f} != TopK {k}"
    return sae, {"fvu": fvu, "l0": l0, "dead_frac": dead, "n_train_tokens": int(n)}

SAES, SAE_HEALTH = {}, {}
for L in LAYERS:
    print(f"\n=== layer {L}: collecting token activations ===")
    TOK = token_acts_for_layer(corpus_seqs, L, MAX_TOK_PER_SEQ)
    print(f"  {TOK.shape[0]} tokens ({TOK.numel() * 2 / 1e9:.2f} GB fp16), "
          f"{TOK.shape[0] / D_LATENT:.0f} per latent")
    print(f"=== layer {L}: training SAE ===")
    SAES[L], SAE_HEALTH[L] = train_sae(TOK, D_LATENT, SAE_TOPK, AUX_K, seed=100 + L)
    h = SAE_HEALTH[L]
    print(f"  layer {L}: FVU {h['fvu']:.4f}, L0 {h['l0']:.1f}, dead {h['dead_frac']:.1%}")
    del TOK
    clear_gpu()

print()
print("HEALTH GATE (pre-registered: FVU < 0.5 AND dead_frac < 0.9)")
for L in LAYERS:
    h = SAE_HEALTH[L]
    ok = h["fvu"] < 0.5 and h["dead_frac"] < 0.9
    print(f"  layer {L}: FVU {h['fvu']:.3f}  dead {h['dead_frac']:.1%}  -> "
          f"{'PASS' if ok else 'FAIL'}")
print()
print("60 for comparison: FVU 0.094/0.117 but dead 98.5%/94.2% -> FAILED on dead fraction")



=== layer 12: collecting token activations ===
    2000/12000 proteins, 123476 tokens
    4000/12000 proteins, 246787 tokens
    6000/12000 proteins, 370413 tokens
    8000/12000 proteins, 494152 tokens
    10000/12000 proteins, 617645 tokens
    12000/12000 proteins, 741030 tokens
  741030 tokens (1.90 GB fp16), 289 per latent
=== layer 12: training SAE ===
    train 721030 tokens / val 20000 held out  (282 tokens per latent)
    epoch 1/10  loss 1626.2088  FVU 0.2955  dead 98.3%
    epoch 2/10  loss 73.6671  FVU 0.1132  dead 98.1%
    epoch 3/10  loss 51.1600  FVU 0.0908  dead 97.4%
    epoch 4/10  loss 37.3267  FVU 0.0598  dead 97.4%
    epoch 5/10  loss 26.7478  FVU 0.0458  dead 97.3%
    epoch 6/10  loss 20.9593  FVU 0.0361  dead 96.4%
    epoch 7/10  loss 16.4935  FVU 0.0278  dead 96.1%
    epoch 8/10  loss 14.0889  FVU 0.0258  dead 95.4%
    epoch 9/10  loss 12.9532  FVU 0.0231  dead 94.3%
    epoch 10/10  loss 12.2030  FVU 0.0225  dead 94.8%
  layer 12: FVU 0.0225, L0 32.0, de

In [6]:
# --- Scoring + ESMFold. Identical to 24/26/27/41, including the length-aware OOM retry added in
#     39/40 and the fold_ok tracking added after the §1l collapse-metric hole. ---
ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq, max_len=300):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False
        working = cleaned[:max_len] if len(cleaned) > max_len else cleaned
        try:
            inputs = self.tokenizer([working], return_tensors="pt", add_special_tokens=False).to(self.device)
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
            return plddt, ptm, True
        except RuntimeError:
            clear_gpu()
        half = max(10, len(working) // 2)
        if half < len(working):
            try:
                inputs = self.tokenizer([working[:half]], return_tensors="pt", add_special_tokens=False).to(self.device)
                with torch.no_grad():
                    out = self.model(**inputs)
                raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
                plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
                ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
                return plddt, ptm, True
            except RuntimeError:
                clear_gpu()
        return 0.0, 0.0, False

def fold_records_ptm(records, evaluator):
    for r in records:
        plddt, ptm, fold_ok = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_ok"] = fold_ok
        r["collapse"] = int(0.0 < plddt < 60.0)
        r["repetition_score"] = repetition_score(r["sequence"])
        r["utility_score"] = utility_score(plddt, ptm) if plddt > 0 else 0.0
    return records

print("Scoring functions and length-aware ESMFold evaluator ready.")


Scoring functions and length-aware ESMFold evaluator ready.


In [7]:
# --- Labelled probe set: same construction and seed as 60, so the two runs are comparable. ---

UNIPROT_ACCESSIONS = ["P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
                      "P00648", "P99999", "P69905", "P68871", "P00698", "P00441"]

def fetch_one(acc):
    try:
        with urllib.request.urlopen(
                f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=15) as r:
            return "".join(r.read().decode("utf-8").strip().split("\n")[1:])
    except Exception:
        return None

refs = [s for s in (fetch_one(a) for a in UNIPROT_ACCESSIONS) if s and len(s) >= 20]
if not refs:
    refs = [s for s in corpus_seqs[:12]]
print(f"{len(refs)} reference proteins for prefixes")

def build_prefix_pool(n, seed=11, lo=10, hi=15):
    r = np.random.RandomState(seed)
    out = []
    for i in range(n):
        s = refs[i % len(refs)]
        L = r.randint(lo, hi + 1)
        st = r.randint(0, max(1, len(s) - L))
        out.append(s[st:st + L])
    return out

print(f"\n=== generating {N_LABELLED} labelled sequences (seed {LABELLED_SEED}, as in 60) ===")
torch.manual_seed(LABELLED_SEED)
labelled = []
for i, p in enumerate(build_prefix_pool(N_LABELLED, seed=LABELLED_SEED)):
    inp = tokenizer(p, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = plm_model.generate(**inp, max_length=50, do_sample=True,
                                 temperature=1.2, pad_token_id=PAD_ID)
    s = tokenizer.decode(ids[0], skip_special_tokens=True).replace(" ", "")
    labelled.append({"prompt": p, "sequence": s, "clean": clean_aa(s),
                     "entropy": calculate_entropy(s)})
    if (i + 1) % 200 == 0:
        print(f"  {i + 1}/{N_LABELLED}")
clear_gpu()

del plm_model
clear_gpu()
evaluator = StructuralEvaluatorPTM()
print("Folding...")
for i, r in enumerate(labelled):
    plddt, ptm, ok_ = evaluator.fold_one(r["sequence"])
    r["plddt"], r["ptm"], r["fold_ok"] = plddt, ptm, ok_
    r["collapse"] = int(0.0 < plddt < 60.0)
    if (i + 1) % 100 == 0:
        done = [q for q in labelled[:i + 1] if q["fold_ok"]]
        print(f"  {i + 1}/{N_LABELLED}  collapse {np.mean([q['collapse'] for q in done]):.1%}")
del evaluator
clear_gpu()

ok = np.array([r["fold_ok"] for r in labelled])
y = np.array([r["collapse"] for r in labelled])[ok]
print(f"\n{ok.sum()}/{N_LABELLED} folded, collapse {y.mean():.1%}  (60 recorded 50.3%)")


12 reference proteins for prefixes

=== generating 800 labelled sequences (seed 70001, as in 60) ===
  200/800
  400/800
  600/800
  800/800
Loading ESMFold...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding...
  100/800  collapse 55.0%
  200/800  collapse 54.0%
  300/800  collapse 51.7%
  400/800  collapse 52.0%
  500/800  collapse 50.0%
  600/800  collapse 51.2%
  700/800  collapse 50.3%
  800/800  collapse 50.2%

800/800 folded, collapse 50.2%  (60 recorded 50.3%)


In [8]:
# --- Features: SAE latents (encode per token, then pool) and raw mean activations. ---

print("Reloading ProtGPT2 for feature extraction...")
plm_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
plm_model.eval()

def features_for_layer(records, layer, sae):
    lat, raw = [], []
    sae = sae.eval()
    for i, r in enumerate(records):
        s = r["clean"] if len(r["clean"]) >= 5 else "MKT"
        enc = tokenizer(s, return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            o = plm_model(**enc, output_hidden_states=True)
            h = o.hidden_states[layer].squeeze(0).float()
            lat.append(sae.encode(h).mean(dim=0).cpu().numpy())
            raw.append(h.mean(dim=0).cpu().numpy())
        if (i + 1) % 200 == 0:
            print(f"  layer {layer}: {i + 1}/{len(records)}")
    return np.vstack(lat), np.vstack(raw)

LATENTS, RAW = {}, {}
for L in LAYERS:
    lat, raw = features_for_layer(labelled, L, SAES[L])
    LATENTS[L], RAW[L] = lat[ok], raw[ok]
    print(f"layer {L}: latents {LATENTS[L].shape}, "
          f"nonzero {float((LATENTS[L] > 0).mean()):.4f}, raw {RAW[L].shape}")
ENT = np.array([r["entropy"] for r in labelled])[ok].reshape(-1, 1)

del plm_model
clear_gpu()


Reloading ProtGPT2 for feature extraction...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  layer 12: 200/800
  layer 12: 400/800
  layer 12: 600/800
  layer 12: 800/800
layer 12: latents (800, 2560), nonzero 0.0173, raw (800, 1280)
  layer 30: 200/800
  layer 30: 400/800
  layer 30: 600/800
  layer 30: 800/800
layer 30: latents (800, 2560), nonzero 0.0325, raw (800, 1280)


In [9]:
# --- The matched comparison. Latent selection fitted INSIDE the fold. ---

class TopKLatentSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k=20):
        self.k = k

    def fit(self, X, y=None):
        y = np.asarray(y)
        d = np.abs(X[y == 1].mean(0) - X[y == 0].mean(0))
        self.idx_ = np.argsort(-d)[: self.k]
        return self

    def transform(self, X):
        return X[:, self.idx_]

def rf(seed):
    return RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)

def evaluate(X, kind, k, seed):
    if kind == "sae":
        pipe = Pipeline([("sel", TopKLatentSelector(k=k)),
                         ("scale", StandardScaler()), ("rf", rf(seed))])
    elif kind == "raw":
        pipe = Pipeline([("scale", StandardScaler()),
                         ("pca", PCA(n_components=k, random_state=seed)), ("rf", rf(seed))])
    else:
        pipe = Pipeline([("scale", StandardScaler()), ("rf", rf(seed))])
    cv = StratifiedKFold(10, shuffle=True, random_state=seed)
    p = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
    return roc_auc_score(y, p)

results, deltas = [], []
for L in LAYERS:
    for k in K_VALUES:
        row = {}
        for kind, X in (("sae", LATENTS[L]), ("raw", RAW[L])):
            a = [evaluate(X, kind, k, 1000 + r) for r in range(N_REPEATS)]
            row[kind] = (float(np.mean(a)), float(np.std(a)))
            results.append({"layer": L, "k": k, "features": kind,
                            "auc_mean": row[kind][0], "auc_sd": row[kind][1]})
        d = row["sae"][0] - row["raw"][0]
        deltas.append(d)
        print(f"  layer {L:2d}  k={k:<3d}  SAE {row['sae'][0]:.4f}  raw {row['raw'][0]:.4f}  "
              f"delta {d:+.4f}")

ent_auc = [evaluate(ENT, "plain", 1, 2000 + r) for r in range(3)]
results.append({"layer": None, "k": 1, "features": "entropy",
                "auc_mean": float(np.mean(ent_auc)), "auc_sd": float(np.std(ent_auc))})
print(f"  entropy baseline AUC {np.mean(ent_auc):.4f}")


  layer 12  k=3    SAE 0.6376  raw 0.6657  delta -0.0281
  layer 12  k=20   SAE 0.7102  raw 0.7605  delta -0.0503
  layer 12  k=64   SAE 0.7443  raw 0.7712  delta -0.0269
  layer 30  k=3    SAE 0.6434  raw 0.6831  delta -0.0397
  layer 30  k=20   SAE 0.7485  raw 0.8337  delta -0.0852
  layer 30  k=64   SAE 0.7892  raw 0.8514  delta -0.0622
  entropy baseline AUC 0.5557


In [10]:
# --- Verdict. ---

BAR = "=" * 92
print(BAR)
print("VERDICT")
print(BAR)
worst_fvu = max(SAE_HEALTH[L]["fvu"] for L in LAYERS)
max_dead = max(SAE_HEALTH[L]["dead_frac"] for L in LAYERS)
gate = worst_fvu < 0.5 and max_dead < 0.9
print(f"SAE health: worst FVU {worst_fvu:.3f}, max dead {max_dead:.1%}  -> "
      f"gate {'PASS' if gate else 'FAIL'}")
print(f"  (60: FVU 0.117, dead 98.5% -> FAIL)")
print()
mean_d = float(np.mean(deltas))
if not gate:
    print("  ==> INCONCLUSIVE AGAIN. The dictionary still is not adequately trained, so the")
    print("      comparison says nothing about SAEs. Report both attempts honestly and STOP --")
    print("      two failures to build an adequate dictionary at workshop scale is itself the")
    print("      result, and the SAE-vs-baselines claim belongs to Kantamneni et al. regardless.")
else:
    print(f"mean (SAE - raw) across {len(deltas)} matched cells: {mean_d:+.4f}")
    if mean_d < -0.02:
        print("  ==> SAE UNDERPERFORMS RAW at matched budget, layer and classifier, with a")
        print("      dictionary that passes its health gate. This is a CONTROLLED REPLICATION of")
        print("      Kantamneni et al. (ICML 2025) in a new domain -- cite them for the direction,")
        print("      claim only the fairness of the test and the protein-domain extension.")
    elif abs(mean_d) <= 0.02:
        print("  ==> SAE AND RAW ARE COMPARABLE at matched budget. Report as a correction of 3a:")
        print("      that gap was the three confounds, not a property of SAE latents.")
    else:
        print("  ==> SAE OUTPERFORMS RAW -- contrary to ICML. Check in-fold selection and that the")
        print("      UniProt corpus is disjoint from the labelled generations before believing it.")

print()
print("Read against 60 (5120 latents, 122k tokens, 98.5% dead):")
print("  60 layer 12: SAE k=3 0.602, k=20 0.690, k=64 0.702")
print("  60 layer 30: SAE k=3 0.538, k=20 0.599, k=64 0.732")
print("  3a (RETRACTED): OSAE 3 latents from a 12-sequence dictionary = 0.528")
print(BAR)


VERDICT
SAE health: worst FVU 0.093, max dead 94.8%  -> gate FAIL
  (60: FVU 0.117, dead 98.5% -> FAIL)

  ==> INCONCLUSIVE AGAIN. The dictionary still is not adequately trained, so the
      comparison says nothing about SAEs. Report both attempts honestly and STOP --
      two failures to build an adequate dictionary at workshop scale is itself the
      result, and the SAE-vs-baselines claim belongs to Kantamneni et al. regardless.

Read against 60 (5120 latents, 122k tokens, 98.5% dead):
  60 layer 12: SAE k=3 0.602, k=20 0.690, k=64 0.702
  60 layer 30: SAE k=3 0.538, k=20 0.599, k=64 0.732
  3a (RETRACTED): OSAE 3 latents from a 12-sequence dictionary = 0.528


In [11]:
# --- Persist. ---

pd.DataFrame(results).to_csv("sae_trained_results.csv", index=False)
pd.DataFrame([{"layer": L, **SAE_HEALTH[L], "d_latent": D_LATENT, "topk": SAE_TOPK,
               "aux_k": AUX_K} for L in LAYERS]).to_csv("sae_trained_health.csv", index=False)
pd.DataFrame([{
    "idx": i, "sequence": r["sequence"], "usable_length": len(r["clean"]),
    "entropy": r["entropy"], "plddt": r["plddt"], "ptm": r["ptm"],
    "fold_ok": r["fold_ok"], "collapse": r["collapse"],
} for i, r in enumerate(labelled)]).to_csv("sae_trained_labelled_sequences.csv", index=False)
pd.DataFrame([{
    "n_uniprot_fetched": len(corpus_seqs), "used_uniprot": USED_UNIPROT,
    "d_latent": D_LATENT, "topk": SAE_TOPK, "aux_k": AUX_K,
    "n_labelled": N_LABELLED, "n_folded": int(ok.sum()), "collapse_rate": float(y.mean()),
    "layers": str(LAYERS), "k_values": str(K_VALUES),
    "mean_sae_minus_raw": mean_d, "health_gate_pass": bool(gate),
    "entropy_auc": float(np.mean(ent_auc)),
}]).to_csv("sae_trained_calibration.csv", index=False)

print("Saved:")
for f in ["sae_trained_results.csv", "sae_trained_health.csv",
          "sae_trained_labelled_sequences.csv", "sae_trained_calibration.csv"]:
    print("  " + f)
print()
print("DOWNLOAD THE OUTPUT TAB BEFORE CLOSING THE SESSION.")
print("sae_trained_health.csv is the one that decides whether anything else here is admissible.")


Saved:
  sae_trained_results.csv
  sae_trained_health.csv
  sae_trained_labelled_sequences.csv
  sae_trained_calibration.csv

DOWNLOAD THE OUTPUT TAB BEFORE CLOSING THE SESSION.
sae_trained_health.csv is the one that decides whether anything else here is admissible.
